In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# Load environment variables
# ============================================================

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "Make sure it exists in your .env file."
    )

client = OpenAI(api_key=api_key)


# ============================================================
# Helper function for OpenAI API calls
# ============================================================

def call_openai(system_prompt, user_prompt, model="gpt-4o"):
    """
    Simple wrapper for OpenAI API calls.
    """

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"An error occurred: {e}"


# ============================================================
# Telecom Agent
# ============================================================

class TelecomAgent:
    """
    Agent that handles Telecom-related questions.
    """

    def __init__(self):

        # Used internally by the router
        self.routing_name = "TelecomAgent"

        # Human-readable name
        self.name = "Telecom Agent"

        self.description = """
        Use this agent for Telecom-related questions.

        Topics include:
        - 3G
        - 4G
        - LTE
        - 5G
        - 5G SA
        - 5G NSA
        - 6G
        - 3GPP
        - RAN
        - Core Network
        - AMF
        - SMF
        - UPF
        - gNB
        - PRACH
        - RRC
        - Mobility
        - Radio procedures
        - Signaling
        """

    def get_description(self):
        return self.description

    def run(self, task_prompt):

        system_prompt = """
        You are a specialist Telecom Agent.

        Your job is to answer telecom questions accurately.

        When appropriate:

        1. Refer to relevant 3GPP specifications.
        2. Use correct 3GPP terminology.
        3. Explain the concept clearly.
        4. Explain important procedures step by step.
        5. Mention relevant network elements.
        6. Distinguish between RAN and Core Network concepts.
        """

        user_prompt = f"""
        Answer the following Telecom question:

        {task_prompt}
        """

        return call_openai(
            system_prompt,
            user_prompt
        )


# ============================================================
# Machine Learning Agent
# ============================================================

class MachineLearningAgent:
    """
    Agent that handles Machine Learning questions.
    """

    def __init__(self):

        self.routing_name = "MachineLearningAgent"

        self.name = "Machine Learning Agent"

        self.description = """
        Use this agent for Machine Learning questions.

        Topics include:
        - Machine Learning models
        - Supervised Learning
        - Unsupervised Learning
        - Reinforcement Learning
        - Regression
        - Classification
        - Clustering
        - Neural Networks
        - Deep Neural Networks
        - Deep Learning
        - Model Training
        - Model Evaluation
        - Feature Engineering
        """

    def get_description(self):
        return self.description

    def run(self, task_prompt):

        system_prompt = """
        You are a specialist Machine Learning Agent.

        Your job is to provide accurate and detailed answers about
        Machine Learning.

        Explain:

        1. Important concepts
        2. Algorithms
        3. Models
        4. Training techniques
        5. Evaluation techniques
        6. Practical examples

        Keep explanations technically accurate but easy to understand.
        """

        user_prompt = f"""
        Answer the following Machine Learning question:

        {task_prompt}
        """

        return call_openai(
            system_prompt,
            user_prompt
        )


# ============================================================
# Artificial Intelligence Agent
# ============================================================

class ArtificialIntelligenceAgent:
    """
    Agent that handles Artificial Intelligence questions.
    """

    def __init__(self):

        self.routing_name = "ArtificialIntelligenceAgent"

        self.name = "Artificial Intelligence Agent"

        self.description = """
        Use this agent for general Artificial Intelligence questions.

        Topics include:
        - Artificial Intelligence
        - AI Agents
        - Agentic AI
        - Reasoning
        - Planning
        - Search Algorithms
        - Knowledge Representation
        - Expert Systems
        - Generative AI
        - Large Language Models
        - Multi-Agent Systems
        - AI Architecture
        """

    def get_description(self):
        return self.description

    def run(self, task_prompt):

        system_prompt = """
        You are a specialist Artificial Intelligence Agent.

        Your job is to provide accurate and detailed answers
        about Artificial Intelligence.

        Explain:

        1. Important AI concepts
        2. AI architectures
        3. Reasoning techniques
        4. Planning techniques
        5. Search algorithms
        6. AI applications
        7. Relevant terminology
        """

        user_prompt = f"""
        Answer the following Artificial Intelligence question:

        {task_prompt}
        """

        return call_openai(
            system_prompt,
            user_prompt
        )


# ============================================================
# Routing Agent
# ============================================================

def routing_agent(task_prompt, agents):
    """
    Uses an LLM to determine which specialist agent
    should handle the user's question.
    """

    # Dynamically generate agent descriptions
    agent_descriptions = "\n".join(
        [
            f"""
Agent Name: {agent.routing_name}
Description:
{agent.get_description()}
"""
            for agent in agents
        ]
    )

    system_prompt = f"""
    You are an AI routing assistant.

    Your ONLY responsibility is to select the most appropriate
    specialist agent for the user's task.

    Available agents:

    {agent_descriptions}

    ------------------------------------------------

    You MUST return exactly ONE of the following values:

    TelecomAgent
    MachineLearningAgent
    ArtificialIntelligenceAgent

    ------------------------------------------------

    Routing guidelines:

    TelecomAgent:
    Use for telecommunications, 3GPP, LTE, 5G, 6G,
    RAN, Core Network, PRACH, AMF, SMF, UPF, gNB,
    radio procedures, mobility, signaling, etc.

    MachineLearningAgent:
    Use for machine-learning concepts such as regression,
    classification, clustering, neural networks,
    deep neural networks, deep learning, model training,
    feature engineering, etc.

    ArtificialIntelligenceAgent:
    Use for broader AI concepts such as AI agents,
    reasoning, planning, search algorithms,
    generative AI, LLMs, multi-agent systems,
    and knowledge representation.

    ------------------------------------------------

    IMPORTANT RULES:

    - Return ONLY the routing name.
    - Do NOT explain your decision.
    - Do NOT use quotes.
    - Do NOT add punctuation.
    - Do NOT return the human-readable agent name.

    Correct example:

    TelecomAgent

    Incorrect examples:

    Telecom Agent
    "TelecomAgent"
    The correct agent is TelecomAgent.
    """

    user_prompt = f"""
    Determine the correct agent for the following task:

    {task_prompt}
    """

    agent_choice_name = call_openai(
        system_prompt,
        user_prompt
    )

    # Remove whitespace
    agent_choice_name = agent_choice_name.strip()

    print(
        f"\nRouter selected: {agent_choice_name}"
    )

    # Find selected agent
    for agent in agents:

        if agent.routing_name == agent_choice_name:

            print(
                f"--- Routing task to "
                f"{agent.name} ---"
            )

            return agent.run(task_prompt)

    return (
        f"Error: Could not find an agent named "
        f"'{agent_choice_name}'."
    )


# ============================================================
# Main Program
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Initialize the agents
    # --------------------------------------------------------

    telecom_agent = TelecomAgent()

    ml_agent = MachineLearningAgent()

    ai_agent = ArtificialIntelligenceAgent()


    # Create list of available agents
    all_agents = [
        telecom_agent,
        ml_agent,
        ai_agent
    ]


    # ========================================================
    # Example 1
    # Telecom question
    # ========================================================

    task1_prompt = (
        "Explain me the concept of PRACH Config Index."
    )

    print("\n" + "=" * 60)

    print(
        f'User Task: "{task1_prompt}"'
    )

    result1 = routing_agent(
        task1_prompt,
        all_agents
    )

    print("\nAgent Output:\n")

    print(result1)


    # ========================================================
    # Example 2
    # Machine Learning question
    # ========================================================

    print("\n" + "=" * 60)

    task2_prompt = (
        "Explain various Machine Learning Models."
    )

    print(
        f'User Task: "{task2_prompt}"'
    )

    result2 = routing_agent(
        task2_prompt,
        all_agents
    )

    print("\nAgent Output:\n")

    print(result2)


    # ========================================================
    # Example 3
    # Deep Learning question
    # ========================================================

    print("\n" + "=" * 60)

    task3_prompt = (
        "What are Deep Neural Networks?"
    )

    print(
        f'User Task: "{task3_prompt}"'
    )

    result3 = routing_agent(
        task3_prompt,
        all_agents
    )

    print("\nAgent Output:\n")

    print(result3)


    # ========================================================
    # Example 4
    # Artificial Intelligence question
    # ========================================================

    print("\n" + "=" * 60)

    task4_prompt = (
        "What are AI agents and how do they perform reasoning?"
    )

    print(
        f'User Task: "{task4_prompt}"'
    )

    result4 = routing_agent(
        task4_prompt,
        all_agents
    )

    print("\nAgent Output:\n")

    print(result4)

    print("\n" + "=" * 60)


User Task: "Explain me the concept of PRACH Config Index."

Router selected: TelecomAgent
--- Routing task to Telecom Agent ---

Agent Output:

The PRACH (Physical Random Access Channel) Config Index is a crucial parameter in LTE (Long Term Evolution) and NR (New Radio, 5G) networks that determines how the random access procedure is configured. This procedure is essential for a User Equipment (UE) to initiate communication with the network, particularly when it first powers on or when it needs to re-establish a connection.

### Key Concepts:

1. **Random Access Procedure**: 
   - This is a process by which a UE requests access to the network. It is used for initial access, handover, and re-establishing connections.
   - The procedure involves the UE sending a preamble to the network, which is a short burst of data that indicates the UE's intention to communicate.

2. **PRACH Config Index**:
   - The PRACH Config Index is a parameter that specifies the configuration of the PRACH in ter